In [ ]:
import os
from pathlib import Path

# Make `mo.image(src="media/foo.png")` resolve regardless of which
# directory `marimo edit` is launched from. We chdir to the notebook's
# own directory so the relative `media/` path works the same way it
# does for the deployed WASM build (where the page sits next to
# `media/`). In WASM, `__file__` may not be set and the chdir becomes
# a no-op — the browser still fetches the images via the page URL.
if "__file__" in globals() and __file__:
    try:
        os.chdir(Path(__file__).resolve().parent)
    except OSError:
        pass

import marimo as mo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Seed once so the same samples back every figure in this notebook —
# otherwise the slides re-roll data on each render and the figures
# change between class sessions.
np.random.seed(0)

# Neural Networks III

## Introduction

- You have now learned about the fundamentals of neural networks.
- The goal of this lecture it to elevate your understanding of neural networks.

### The core idea of neural networks - transform data into a new representation.

- Neural networks belong to the field of research called representation learning.
- Very useful in cases where the original representation is difficult to work with.
    - Images, text, graphs, etc.

In [ ]:
# Two noisy concentric circles in Cartesian space and the same
# samples in polar coordinates. The polar view collapses each
# circle onto a thin horizontal band of radii, making the two
# classes trivially separable — a concrete illustration of
# representation learning.
n_samples_polar = 200

r_inner_polar = 1
theta_inner_polar = 2 * np.pi * np.random.rand(n_samples_polar // 2)
x_inner_polar = r_inner_polar * np.cos(theta_inner_polar) + 0.2 * np.random.randn(n_samples_polar // 2)
y_inner_polar = r_inner_polar * np.sin(theta_inner_polar) + 0.2 * np.random.randn(n_samples_polar // 2)
r_inner_polar_calc = np.sqrt(x_inner_polar ** 2 + y_inner_polar ** 2)
theta_inner_polar_calc = np.arctan2(y_inner_polar, x_inner_polar)

r_outer_polar = 2.5
theta_outer_polar = 2 * np.pi * np.random.rand(n_samples_polar // 2)
x_outer_polar = r_outer_polar * np.cos(theta_outer_polar) + 0.2 * np.random.randn(n_samples_polar // 2)
y_outer_polar = r_outer_polar * np.sin(theta_outer_polar) + 0.2 * np.random.randn(n_samples_polar // 2)
r_outer_polar_calc = np.sqrt(x_outer_polar ** 2 + y_outer_polar ** 2)
theta_outer_polar_calc = np.arctan2(y_outer_polar, x_outer_polar)

fig_polar, (ax_polar_xy, ax_polar_rt) = plt.subplots(1, 2, figsize=(12, 4))
ax_polar_xy.scatter(x_inner_polar, y_inner_polar, color="blue", edgecolor="k",
                    s=80, label="Class 1")
ax_polar_xy.scatter(x_outer_polar, y_outer_polar, color="red", edgecolor="k",
                    s=80, label="Class 2")
ax_polar_xy.set_xlabel("x")
ax_polar_xy.set_ylabel("y")
ax_polar_xy.set_aspect("equal")
ax_polar_xy.legend()
ax_polar_rt.scatter(r_inner_polar_calc, theta_inner_polar_calc, color="blue",
                    edgecolor="k", s=80, label="Class 1")
ax_polar_rt.scatter(r_outer_polar_calc, theta_outer_polar_calc, color="red",
                    edgecolor="k", s=80, label="Class 2")
ax_polar_rt.set_xlabel("r")
ax_polar_rt.set_ylabel("θ")
ax_polar_rt.legend()
fig_polar.tight_layout()

mo.as_html(fig_polar)
plt.close(fig_polar)

### The core idea of neural networks - transform data into a new representation

- Show video
- [Visualizing Neural Networks with the Grand Tour](https://distill.pub/2020/grand-tour/)

## From scalars and sum to vectors and matrix multiplication

- Our derivation of Backpropagation two lectures back looked at scalar derivatives.
- Very little linear algebra and vector calculus.
- Simple operation, but leads to complicated computations
    - A lot of indices and sums.
- We will now continue the representation perspective to derive a much cleaner version of Backpropagation.
    - Bonus: also aligns much more closely to how the code should look!

### A new perspective of neural networks - modules of computation

- Draw example.

### Forward pass with linear algebra

- Before we looked at the pre-activation of one neuron: $z_j^l = \mathbf{w}_j^l \mathbf{a}^{l-1}$
- Note: augmented space!
- Now we look at the pre-activations of the whole layer: $\mathbf{z}^l =$

---

- Reminder: $\mathbf{a}^{l-1} =$

---

- And : $\mathbf{W}^l =$



### Forward pass with linear algebra

- Usually want to process a batch of samples.
- Can also be done efficiently!
- Let a batch of inputs be represented as $\mathbf{A}^{l-1} = \begin{bmatrix} a_{1,1}^{l-1} & a_{1,2}^{l-1} & \cdots & a_{1,k_{l-1}}^{l-1} & 1 \\ a_{2,1}^{l-1} & a_{2,2}^{l-1} & \cdots & a_{2,k_{l-1}}^{l-1} & 1 \\ \vdots        & \vdots        & \ddots & \vdots              \\ a_{N,1}^{l-1} & a_{N,2}^{l-1} & \cdots & a_{N,k_{l-1}}^{l-1} & 1 \end{bmatrix}$

---

- Then:

---

### Backward pass with linear algebra and vector calculus

- Previously, for the output layer: $\frac{\partial}{\partial \mathbf{w}_j^L} E (i) = \frac{\partial}{\partial \mathbf{w}_j^L} z_j^L (i) \frac{\partial}{\partial z_j^L (i) }E (i)$
- Want: $\frac{\partial J}{\partial \mathbf{W}^l}$
- Difficult, ends up with a vector by matrix derivate.
- Start simpler: $\frac{\partial}{\partial \mathbf{w}_j^L} \mathbf{z}^L \frac{\partial}{\partial \mathbf{z}^L}\mathbf{e}^T\mathbf{e}\frac{1}{2}$
- Where we assume one-hot encoded labels and $\mathbf{e}=(\mathbf{a}^L-\mathbf{y})$

### Finding $\delta$ for output layer

- We have dealt with the following term before $\frac{\partial}{\partial \mathbf{z}^L}\mathbf{e}^T\mathbf{e}\frac{1}{2}=\mathbf{e}\frac{\partial}{\partial \mathbf{z}^L}(\mathbf{a}^L-\mathbf{y})$
- $\mathbf{y}$ does not depend on $\mathbf{z}^L$, and we keep the derivative of $\mathbf{a}^L=f(\mathbf{z}^L)$ general.
- We have a vector by vector derivative $\Rightarrow$ Jacobian: $\frac{\partial \mathbf{a}^L}{\partial \mathbf{z}^L} =$

---

### Finding $\delta$ for output layer

- Can be compatctly represented as $\frac{\partial}{\partial \mathbf{z}^L}\mathbf{e}^T\mathbf{e}\frac{1}{2} =$

---

- $\odot$ is the Hadamard product or elementwise multiplication.

### Backward pass with linear algebra and vector calculus

- Now, we turn to: $\frac{\partial}{\partial \mathbf{w}_j^L} \mathbf{z}^L$
- Vector by vector derivatve $\Rightarrow$ Jacobian matrix:

---

### Derivative of one element in Jacobian

- Reminder: $z^L_1 = w_{1,1}*a^{L-1}_1 + w_{1,2}*a^{L-1}_2 + \cdots w_{1,k_{L-1}}*a^{L-1}_{k_{L-1}} + 1*w_{1,0}$
- Therefore: $\frac{\partial}{\partial w_{1,1}}z^L_1 =$

---

- Back to Jacobian:

### Putting it all together

- Combing both derivatives give:

---

### Putting it all together

- Take a step back. Derivative of loss with respect to neuron 1 gave non-zero elements in column 1.
- If we repeat process of derivative with resepct to neuron $j$, coulmn $j$ will be non-zero.
- We can get all derivatives with one matrix operation:
- $\frac{\partial J}{\partial \mathbf{W}^L} =$

---

### What have we gained?

- A lot of work, but clean result with simple rule.
- To find gradients of loss with respect to weights in layer $l$, look at $\delta$ from current layer and output from previous layer.

## A final touch on neural networks

- Many nice results for linear classifiers.
    - Optimal in terms of e.g. maximum likelihood.
    - Guarantueed unique solution for SVM.
- Hard to obtain similar results for neural networks due to their complexity.
- However, work is ongoing!

### Connecting SVMs and neural networks

- The big question in neural network research:
    - Why do they generalize so well?
    - Seems to defy traditional statistics.
- One of many directions to answer this question:
    - Generalization due to implicit bias in gradient descent algorithm.
- [Interesting work by Soundry et al (2018)](https://www.jmlr.org/papers/volume19/18-188/18-188.pdf)

In [ ]:
mo.vstack(
    [
        mo.md(
            r"""
### Connecting SVMs and neural networks

- Setting:
    - Neural network with one layer.
    - Binary classification.
    - Separable classes (similar results have been shown for non-separable classes).
    - Also some assumptions on loss function.
            """
        ),
        # Setting figure from Soudry et al. — the optimisation
        # trajectory in the (w₁, w₂) plane that motivates the
        # gradient-descent / hard-margin-SVM equivalence below.
        mo.image(src="media/soundry.png", width="700px"),
    ],
    gap=1,
)


In [ ]:
mo.vstack(
    [
        mo.md(
            r"""
### Connecting SVMs and neural networks

- What happens when we let the number of iterations tend towards infinity?
- Remarkably, we obtain the weights of the hard margin SVM!
- They also show how the KKT conditions "naturally" fall out of this analysis.
            """
        ),
        # Companion figure — same setting after many iterations; the
        # weight vector aligns with the max-margin SVM solution and
        # the KKT-supporting samples are highlighted on the data.
        mo.image(src="media/soundry2.png", width="700px"),
    ],
    gap=1,
)


## Wrapping up neural networks

- Neural networks are powerful and versatile tools.
- You have now learned the fundamentals. A lot more to be said.
    - However, this foundation often applies also for bigger and more modern neural networks.
- This course is your chance to code a neural network from scratch, take that opportunity!